In [ ]:
# ==============================================================================
# IMPORTS AND SETUP
# ==============================================================================
# This notebook computes Conservation Diversity Entropy (CDE) for natural MSA 
# sequences using multiple DCA models and generates violin plots for comparison
# ==============================================================================

import sys
import os
from pathlib import Path
import gc

# Third-party libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# adabmDCA package imports
from adabmDCA.fasta import get_tokens, import_from_fasta
from adabmDCA.utils import get_device, get_dtype
from adabmDCA.functional import one_hot
from adabmDCA.io import load_params
from adabmDCA.statmech import compute_energy

print("All libraries imported successfully!")
output_path = "images/" 
os.makedirs(output_path, exist_ok=True)

In [ ]:
# ==============================================================================
# PLOT STYLING CONFIGURATION
# ==============================================================================
# Configure color palette and matplotlib style for publication-quality figures
# ==============================================================================

# Define color palette for each DCA model variant
palette = sns.color_palette("deep", 4)
colors = {
    'meDCA': palette[0],    # Medium entropy DCA - Blue
    'edDCA': palette[1],    # Extreme density DCA - Purple-Pink
    'bmDCA': palette[2],    # Boltzmann machine DCA - Pink-Orange
    'eaDCA': palette[3]     # Extreme attributes DCA - Red
}

# Configure LaTeX-style plotting aesthetics
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.frameon": False
})

print("Plot styling configured!")

In [ ]:
# ==============================================================================
# DEVICE, PATHS, AND TOKENS CONFIGURATION
# ==============================================================================

# Configure computational devices
device = get_device("cpu")          # CPU device for general operations
device2 = get_device("cuda")        # GPU device for intensive CDE computation
dtype = get_dtype("float32")         # Data type for tensor operations

# Data paths
data_path = "data/CM_130530_MC.fasta"
weights_path = "data/weights.dat"

# Model paths - different DCA model variants trained on the same MSA
HAmodel_path = "models/Parameters_edDCA_density=0.125_high_entropy.dat" 
decmodel_path = "models/Parameters_conv_dec_new.dat"
eamodel_path = "models/Parameters_EAALG0.8_new.dat"
densemodel_path = "models/Parameters_conv_dense_new.dat"

# Define amino acid tokens (20 amino acids + gap character)
tokens = get_tokens("ACDEFGHIKLMNPQRSTVWY-")

print("Configuration loaded successfully!")
print(f"Using devices: {device} and {device2}")
print(f"Number of tokens: {len(tokens)}")

In [ ]:
# ==============================================================================
# LOAD NATURAL MULTIPLE SEQUENCE ALIGNMENT (MSA)
# ==============================================================================
# Import sequences from FASTA file and convert to one-hot encoding
# ==============================================================================

print("Loading natural MSA...")
headers, msa_enc_nat = import_from_fasta(
    data_path, 
    tokens=tokens, 
    filter_sequences=True  # Remove sequences with invalid characters
)

print(f"MSA shape: {msa_enc_nat.shape}")
print(f"Number of sequences: {len(headers)}")

# Convert to PyTorch tensor
msa_enc_nat = torch.tensor(msa_enc_nat, device=device, dtype=torch.long)

# Convert to one-hot encoding: (num_sequences, L, q)
# where L = sequence length, q = number of tokens
msa_oh_nat = one_hot(msa_enc_nat, num_classes=len(tokens))
print(f"One-hot MSA shape: {msa_oh_nat.shape}")

print("Natural MSA loaded successfully!")

In [ ]:
# ==============================================================================
# LOAD SEQUENCE WEIGHTS AND SAMPLE INDICES
# ==============================================================================
# Load precomputed sequence weights and sample indices for analysis
# Sampling reduces computational burden while maintaining statistical properties
# ==============================================================================

# Load sequence weights (typically computed based on sequence identity)
weights = torch.from_numpy(np.loadtxt(weights_path)).float()
print(f"Weights shape: {weights.shape}")

# Sample indices according to the weight distribution
# This ensures that representative sequences are selected for CDE analysis
num_samples = 1000
probs = weights / weights.sum()  # Normalize to probability distribution
sampled_indices = torch.multinomial(probs, num_samples=num_samples, replacement=True)

print(f"Sampled {num_samples} indices according to sequence weights")
print(f"Sampled indices shape: {sampled_indices.shape}")

In [ ]:
# ==============================================================================
# LOAD DCA MODELS
# ==============================================================================
# Load four different DCA model variants for comparison:
# - meDCA: Medium entropy DCA model
# - edDCA: Extreme density DCA model  
# - bmDCA: Boltzmann machine DCA model
# - eaDCA: Extreme attributes DCA model
# ==============================================================================

print("Loading DCA models...")

original_tokens = get_tokens("ACDEFGHIKLMNPQRSTVWY-")

# Load all four model variants
HA_model = load_params(HAmodel_path, tokens=original_tokens, device=device, dtype=dtype)
dec_model = load_params(decmodel_path, tokens=original_tokens, device=device, dtype=dtype)
ea_model = load_params(eamodel_path, tokens=original_tokens, device=device, dtype=dtype)
dense_model = load_params(densemodel_path, tokens=original_tokens, device=device, dtype=dtype)

print("All models loaded successfully!")
print(f"  - meDCA (HA) model")
print(f"  - edDCA (dec) model")
print(f"  - eaDCA model")
print(f"  - bmDCA (dense) model")

In [ ]:
# ==============================================================================
# CDE COMPUTATION FUNCTION
# ==============================================================================
# Conservation Diversity Entropy (CDE) measures the mutability at each position
# by computing the entropy of the probability distribution over all amino acids
# when considering single mutations at that position.
#
# Algorithm:
# 1. For each sequence, generate all possible single-point mutations
# 2. Compute energy for each mutant using the DCA model
# 3. Calculate probability distribution: p_i = exp(-E_i) / Z
# 4. Compute entropy: CDE = -sum(p_i * log2(p_i))
#
# Returns CDE values for each position in each sequence
# ==============================================================================

def compute_CDE_full_batch_fast(sequences, model, device=device, dtype=dtype):
    """
    Compute Conservation Diversity Entropy for all sequences in batch.
    
    Parameters:
    -----------
    sequences : torch.Tensor
        One-hot encoded sequences of shape (batch_size, L, q)
    model : dict
        DCA model parameters containing 'bias' and 'coupling_matrix'
    device : torch.device
        Device to use for computation (CPU or CUDA)
    dtype : torch.dtype
        Data type for tensor operations
        
    Returns:
    --------
    CDE_full_batch : torch.Tensor
        CDE values of shape (batch_size, L) on CPU
    """
    
    with torch.no_grad():  # Memory efficient - no gradient computation needed
        batch_size, L, q = sequences.shape
        CDE_full_batch = torch.zeros((batch_size, L), device=device, dtype=dtype)
 
        # Process in chunks to manage GPU memory
        chunk_size = 150
        for start in range(0, batch_size, chunk_size):
            end = min(start + chunk_size, batch_size)
            sequences_chunk = sequences[start:end]
            chunk_batch_size = sequences_chunk.shape[0]
            
            # Move model parameters to the target device
            model_device = {}
            model_device["bias"] = model["bias"].to(device)
            model_device["coupling_matrix"] = model["coupling_matrix"].to(device)
            
            # Get amino acid indices from one-hot encoding
            sequence_indices = sequences_chunk.argmax(-1)
            
            # Create base tensor for all possible single mutations
            # Shape: (chunk_batch_size * L * q, L)
            base = sequence_indices.unsqueeze(1).expand(-1, L * q, -1).reshape(chunk_batch_size * L * q, L).clone().to(device)
            
            # Generate position and value arrays for mutations
            positions = torch.arange(L, device=device).repeat_interleave(q).repeat(chunk_batch_size)
            values = torch.arange(q, device=device, dtype=torch.int64).repeat(L).repeat(chunk_batch_size)
            
            # Apply mutations to create Deep Mutational Scan (DMS) sequences
            base[torch.arange(chunk_batch_size * L * q, device=device), positions] = values
            dms = base
            
            # Convert to one-hot encoding for energy computation
            dms_oh = one_hot(dms, num_classes=q).to(dtype)
            
            # Compute energy for all mutants
            energy_dms = compute_energy(dms_oh, model_device)
            energy_dms = energy_dms.view(chunk_batch_size, L, q)
            
            # Normalize energies (shift minimum to zero for numerical stability)
            energy_dms -= energy_dms.min(dim=2, keepdim=True)[0]
            
            # Compute Boltzmann probabilities: p_i = exp(-E_i) / Z
            exp_energy_dms = torch.exp(-energy_dms)
            sums = exp_energy_dms.sum(dim=2, keepdim=True)
            p_i = exp_energy_dms / sums
            
            # Compute Shannon entropy as CDE measure
            CDE_chunk = -torch.sum(p_i * torch.log2(p_i + 1e-10), dim=2)  # Add small epsilon to avoid log(0)
            CDE_full_batch[start:end] = CDE_chunk.cpu()
            
            # Free GPU memory explicitly
            del base, positions, values, dms, dms_oh, energy_dms, exp_energy_dms, sums, p_i
            del model_device["bias"], model_device["coupling_matrix"], CDE_chunk
            
        # Clean up
        del sequences
        gc.collect()
        torch.cuda.empty_cache()
        
    return CDE_full_batch.cpu()

print("CDE computation function defined!")

In [ ]:
# ==============================================================================
# COMPUTE CDE FOR ALL MODELS
# ==============================================================================
# Compute CDE values for natural MSA sequences using each DCA model
# CDE quantifies position-specific mutability as predicted by each model
# ==============================================================================

print("Computing CDE for all models...")
print("This may take several minutes depending on MSA size and GPU availability...")

# Move data to GPU for faster computation
msa_oh_nat_gpu = msa_oh_nat.to(device2)

# Initialize storage for results
CDE_results = {}
CDE_mean_results = {}

# Create model dictionary with descriptive names
models = {
    'meDCA': HA_model,      # Medium entropy model
    'edDCA': dec_model,     # Extreme density model
    'eaDCA': ea_model,      # Extreme attributes model
    'bmDCA': dense_model    # Boltzmann machine model
}

# Compute CDE for each model
for model_name, model in models.items():
    print(f"\n{model_name} model:")
    
    # Compute CDE for all positions and sequences
    CDE = compute_CDE_full_batch_fast(msa_oh_nat_gpu, model, device=device2)
    print(f"  CDE shape: {CDE.shape}")
    
    # Compute mean CDE across all positions for each sequence
    # This gives a single mutability score per sequence
    CDE_mean = CDE.mean(-1)
    print(f"  CDE mean shape: {CDE_mean.shape}")
    print(f"  Average CDE value: {CDE_mean.mean().item():.4f}")
    
    # Store results
    CDE_results[model_name] = CDE
    CDE_mean_results[model_name] = CDE_mean

print("\n" + "="*60)
print("All CDE computations completed successfully!")
print("="*60)

In [ ]:
# ==============================================================================
# VIOLIN PLOT - ALL FOUR MODELS
# ==============================================================================
# Create violin plot comparing CDE distributions across all four DCA models
# Using weighted sampling to ensure representative sequences
# ==============================================================================

# Define model order for plot (low to high entropy)
order = ['eaDCA', 'bmDCA', 'edDCA', 'meDCA']

# Extract sampled CDE mean values for each model
sampled_cde_means = [CDE_mean_results[name].cpu().numpy()[sampled_indices] for name in order]
model_names = order

# Create violin plot
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=sampled_cde_means, 
    inner="box",  # Show box plot inside violin
    palette=[colors[name] for name in model_names]
)

# Configure plot aesthetics
plt.xticks(ticks=range(len(model_names)), labels=model_names, fontsize=24)
plt.ylabel('Mutability (CDE)', fontsize=26)
plt.yticks(fontsize=24)
plt.title('')
plt.grid(True, alpha=0.4)
plt.tight_layout()

# Save figure
plt.savefig(output_path + "violin_plot_CDE_mean_ALL.png", dpi=300, bbox_inches='tight')
print("Saved: violin_plot_CDE_mean_ALL.png")

plt.show()

# Print summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS - ALL MODELS")
print("="*60)
for name in order:
    values = CDE_mean_results[name].cpu().numpy()[sampled_indices]
    print(f"\n{name}:")
    print(f"  Mean ± Std: {values.mean():.4f} ± {values.std():.4f}")
    print(f"  Range: [{values.min():.4f}, {values.max():.4f}]")

In [ ]:
# ==============================================================================
# VIOLIN PLOT - eaDCA vs meDCA COMPARISON
# ==============================================================================
# Create violin plot comparing only the extreme cases:
# - eaDCA: Extreme attributes model (typically lower CDE)
# - meDCA: Medium entropy model (typically higher CDE)
# ==============================================================================

# Select only eaDCA and meDCA models
order = ['eaDCA', 'meDCA']

# Extract sampled CDE mean values for selected models
sampled_cde_means = [CDE_mean_results[name].cpu().numpy()[sampled_indices] for name in order]
model_names = order

# Create violin plot
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=sampled_cde_means, 
    inner="box",  # Show box plot inside violin
    palette=[colors[name] for name in model_names]
)

# Configure plot aesthetics
plt.xticks(ticks=range(len(model_names)), labels=model_names, fontsize=24)
plt.ylabel('Mutability (CDE)', fontsize=26)
plt.yticks(fontsize=24)
plt.title('')
plt.grid(True, alpha=0.4)
plt.tight_layout()

# Save figure
plt.savefig(output_path + "violin_plot_CDE_mean.png", dpi=300, bbox_inches='tight')
print("Saved: violin_plot_CDE_mean.png")

plt.show()

# Print summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS - eaDCA vs meDCA")
print("="*60)
for name in order:
    values = CDE_mean_results[name].cpu().numpy()[sampled_indices]
    print(f"\n{name}:")
    print(f"  Mean ± Std: {values.mean():.4f} ± {values.std():.4f}")
    print(f"  Range: [{values.min():.4f}, {values.max():.4f}]")